# Prepare Google SCIN data for this project

This notebook downloads SCIN images using the official Google Cloud Storage access method, displays the real dermatologist labels, and creates a `train/` + `val/` ZIP for `COLAB_TRAIN_INDIAN_SKIN.ipynb`.

Important: SCIN does not provide the project's `Normal Skin` or `Other Non Skin` classes. Do not invent those labels. Add separately verified healthy-skin and non-skin images. SCIN is also not Maharashtra-specific; add consented, dermatologist-reviewed Indian images for that goal.

In [ ]:
!pip -q install google-cloud-storage pandas pillow scikit-learn

import io, json, os, re, shutil, zipfile
from pathlib import Path
from io import BytesIO
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from google.colab import auth
from google.cloud import storage

auth.authenticate_user()
print('Google authentication completed.')

In [ ]:
# Official SCIN locations and label columns.
PROJECT = 'dx-scin-public'
BUCKET = 'dx-scin-public-data'
CASES_CSV = 'dataset/scin_cases.csv'
LABELS_CSV = 'dataset/scin_labels.csv'
IMAGE_COLUMNS = ['image_1_path', 'image_2_path', 'image_3_path']
LABEL_COLUMN = 'weighted_skin_condition_label'
DERMATOLOGIST_LABEL_COLUMN = 'dermatologist_skin_condition_on_label_name'

client = storage.Client(project=PROJECT)
bucket = client.bucket(BUCKET)
def read_csv_from_bucket(path):
    return pd.read_csv(io.BytesIO(bucket.blob(path).download_as_bytes()), dtype={'case_id': str})

cases = read_csv_from_bucket(CASES_CSV)
labels = read_csv_from_bucket(LABELS_CSV)
df = cases.merge(labels, on='case_id', how='inner')
print('Cases:', len(df))
print('Images referenced:', sum(df[c].notna().sum() for c in IMAGE_COLUMNS))
print('Available SCIN labels:')
print(df[LABEL_COLUMN].value_counts(dropna=False).to_string())

## Choose labels carefully

Look at the previous cell's output. Put the **exact SCIN label text** on the left side below. The code intentionally stops if a label is not present, so labels are not silently guessed.

Example only (replace these keys after seeing the output):

In [ ]:
# Format: exact SCIN label -> project folder name
SCIN_LABEL_MAP = {
    # These exact labels are visible in the SCIN output.
    "{'Acne': 1.0}": 'Acne Vulgaris',
    "{'Eczema': 1.0}": 'Eczema',
    "{'Psoriasis': 1.0}": 'Psoriasis',
    "{'Tinea': 1.0}": 'Fungal Infection',
    "{'Tinea Versicolor': 1.0}": 'Fungal Infection',
    # Add Scabies/Vitiligo only if the exact pure labels appear above.
    # "{'Scabies': 1.0}": 'Scabies',
    # "{'Vitiligo': 1.0}": 'Vitiligo',
}

if not SCIN_LABEL_MAP:
    raise ValueError('Fill SCIN_LABEL_MAP with exact labels from the previous cell, then run this cell again.')
available = set(df[LABEL_COLUMN].dropna().astype(str))
unknown = sorted(set(SCIN_LABEL_MAP) - available)
if unknown:
    raise ValueError(f'These labels are not present exactly in SCIN: {unknown}')

selected = df[df[LABEL_COLUMN].isin(SCIN_LABEL_MAP)].copy()
selected['project_label'] = selected[LABEL_COLUMN].map(SCIN_LABEL_MAP)
print(selected.groupby('project_label')['case_id'].nunique().to_string())
print('Selected cases:', selected['case_id'].nunique())

## Download and split by case

All images from one SCIN case stay in the same split. This prevents nearly identical images from appearing in both training and validation.

In [ ]:
OUTPUT = Path('/content/scin_training_dataset')
if OUTPUT.exists():
    shutil.rmtree(OUTPUT)

case_labels = selected[['case_id', 'project_label']].drop_duplicates()
train_cases, val_cases = train_test_split(
    case_labels, test_size=0.20, random_state=42, stratify=case_labels['project_label']
)
split_by_case = dict.fromkeys(train_cases['case_id'], 'train')
split_by_case.update(dict.fromkeys(val_cases['case_id'], 'val'))

def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]', '_', str(value))

downloaded = 0
failed = []
for _, row in selected.iterrows():
    split = split_by_case[row['case_id']]
    class_dir = OUTPUT / split / row['project_label']
    class_dir.mkdir(parents=True, exist_ok=True)
    for image_column in IMAGE_COLUMNS:
        image_path = row[image_column]
        if not isinstance(image_path, str):
            continue
        try:
            image_bytes = bucket.blob(image_path).download_as_bytes()
            image = Image.open(BytesIO(image_bytes)).convert('RGB')
            filename = safe_name(f"{row['case_id']}_{image_column}.jpg")
            image.save(class_dir / filename, quality=95)
            downloaded += 1
        except Exception as error:
            failed.append((image_path, str(error)))

print('Downloaded images:', downloaded)
print('Failed images:', len(failed))
for item in failed[:10]:
    print(item)
print('Output:', OUTPUT)

In [ ]:
# Check counts and create the ZIP accepted by COLAB_TRAIN_INDIAN_SKIN.ipynb.
for split in ['train', 'val']:
    print(split)
    for class_dir in sorted((OUTPUT / split).glob('*')):
        count = len(list(class_dir.glob('*.jpg')))
        print(f'  {class_dir.name}: {count}')

zip_path = Path('/content/scin_training_dataset.zip')
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for file_path in OUTPUT.rglob('*'):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(OUTPUT.parent))
print('Created:', zip_path)

from google.colab import files
files.download(str(zip_path))